# SIFE-LDM V2 — Colab TPU-Accelerated Training Notebook

**Architecture:** Complex-field diffusion with Phase-Routed MoE, Token Merging, Learnable Meta-Physics, Continuous-time SDE, and Phase-Coherent CFG conditioning.

**Dataset:** CIFAR-100 (100-class, 32×32 images)

**TPU:** Configured for **Colab TPU v2-8** Colab GPUs with automatic VRAM-based scaling.

**Setup Modes:**
- **Antigravity Extension** — files auto-synced from your local workspace
- **Google Drive** — upload the project folder to Drive and mount it
- **Direct Upload** — upload a zip of the project to Colab

**Outputs:** Class-conditional image generation via `cfg_guided_sample()`

## Cell 1: Workspace Setup & Environment
Detects whether you're using the Antigravity Extension, Google Drive, or a direct upload, and sets up the working directory accordingly.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# --- 1. Install Missing Dependencies ---
print('📦  Installing/Verifying required libraries...')
try:
    import flax, optax
except ImportError:
    os.system('pip install -q flax optax')
    print('✅  Libraries installed.')

# --- 2. Workspace Detection & Sync ---
WORKING_DIR = '/content/sife-ldm'
GITHUB_REPO = 'https://github.com/vicekha/SIFE-LDM-.git'

def find_project_root(search_dir):
    for root, dirs, files in os.walk(search_dir):
        if 'scripts' in dirs and 'sife' in dirs:
            return root
    return None

# Check if we are in an Antigravity synced environment
if os.path.exists('/content/sife') and os.path.exists('/content/scripts'):
    WORKING_DIR = '/content'
    print('✅  Antigravity Extension detected.')
else:
    # Try to find or clone the repo
    found_root = find_project_root('/content')
    if found_root:
        WORKING_DIR = found_root
        print(f'📂  Found project at: {WORKING_DIR}')
        os.chdir(WORKING_DIR)
        print('🔄  Syncing latest changes from GitHub...')
        os.system('git pull origin main')
    else:
        print(f'💡  Cloning project from {GITHUB_REPO}...')
        os.system(f'git clone {GITHUB_REPO} {WORKING_DIR}')
        os.chdir(WORKING_DIR)

os.environ['PYTHONPATH'] = os.getcwd()
print(f'\n📂  Working directory set to: {os.getcwd()}')
print(f'📄  Files in root: {os.listdir(".")[:12]}...')

# Verify critical files
critical = ['sife/model.py', 'scripts/train_vision_gpu.py', 'scripts/download_cifar.py']
missing = [f for f in critical if not os.path.exists(f)]
if missing:
    print(f'\n❌  CRITICAL ERROR: Missing files: {missing}')
else:
    print('\n✅  Environment ready for SIFE-LDM training.')

## Cell 2: TPU Initialization & Auto-Scaling
Detects your Colab TPU and automatically configures optimal training parameters.

In [ ]:
import os, sys, json

# Prevent JAX from pre-allocating all memory
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

import jax
import jax.numpy as jnp

print(f'Python: {sys.version}')
print(f'JAX:    {jax.__version__}')

# --- TPU Initialization & Auto-Scaling ---
def setup_tpu_and_scale():
    """Detect Colab TPU type and return optimal training params."""
    try:
        import jax.tools.colab_tpu
        jax.tools.colab_tpu.setup_tpu()
    except KeyError:
        # In case we're not actually in the Colab TPU environment or already initialized
        pass
    except ImportError:
        print('⚠️ jax.tools.colab_tpu not found. Ensure you are in a Colab TPU runtime.')
        
    devices = jax.devices()
    print(f'Devices: {devices}')
    
    tpu_devices = [d for d in devices if d.platform == 'tpu']
    
    if not tpu_devices:
        print('⚠️  No TPU detected! Training will be extremely slow.')
        print('    Go to Runtime → Change runtime type → TPU')
        return {'batch_size': 4, 'embed_dim': 64, 'base_features': 32,
                'max_steps': 10000, 'accel_name': 'CPU', 'num_cores': 0}

    num_cores = len(tpu_devices)
    tpu_name = str(tpu_devices[0])
    print(f'\n🖥️  TPU Device: {tpu_name} (Total Cores: {num_cores})')

    # Auto-scale based on Colab TPU v2-8 specifications (typically 8 cores, 64GB total memory)
    if num_cores >= 8:
        cfg = {'batch_size': 128, 'embed_dim': 256, 'base_features': 128,
               'max_steps': 100000, 'accel_name': 'TPU v2-8', 'num_cores': num_cores}
        print('🚀  TPU v2-8 — MAXIMUM config')
    elif num_cores >= 4:
        cfg = {'batch_size': 64, 'embed_dim': 192, 'base_features': 96,
               'max_steps': 75000, 'accel_name': 'TPU v2-4', 'num_cores': num_cores}
        print('⚡  TPU 4-core — MEDIUM-HIGH config')
    else:
        cfg = {'batch_size': 32, 'embed_dim': 128, 'base_features': 64,
               'max_steps': 50000, 'accel_name': f'TPU ({num_cores} cores)', 'num_cores': num_cores}
        print(f'💡  Smaller TPU ({num_cores} cores) — BASE config')

    print(f'\n📋  Training Config:')
    for k, v in cfg.items():
        print(f'    {k:15s} = {v}')
    return cfg

ACCEL_CONFIG = setup_tpu_and_scale()

# Save for downstream cells
with open('./accel_config.json', 'w') as f:
    json.dump(ACCEL_CONFIG, f, indent=2)
print(f'\n✅  Config saved to ./accel_config.json')

# Verify V2 architecture files
print('\n--- V2 Architecture Check ---')
import os; os.system("grep -l 'LabelEncoder\|EulerMaruyamaSampler\|PhasePool\|predict_meta_physics' sife/model.py sife/diffusion.py sife/unet.py 2>/dev/null")


## Cell 3: CIFAR-100 Data Download
Downloads and prepares the CIFAR-100 dataset.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

import os; os.system("python scripts/download_cifar.py")

import numpy as np
images = np.load('./data/cifar100/train_images.npy')
labels = np.load('./data/cifar100/train_labels.npy')
print(f'Dataset loaded: {images.shape} images, {labels.shape} labels')
print(f'Image range: [{images.min():.3f}, {images.max():.3f}]')

## Cell 4: TPU-Optimized V2 Training

Training parameters are **automatically scaled** based on the TPU config detected in Cell 2.

| TPU Tier  | Batch | Embed Dim | Base Feat | Max Steps |
|-----------|-------|-----------|-----------|-----------|
| TPU v2-8  | 128   | 256       | 128       | 100,000   |
| TPU 4-core| 64    | 192       | 96        | 75,000    |
| Smaller   | 32    | 128       | 64        | 50,000    |

**V2 Enhancements:**
- `ImageEncoder` — learned RGB→complex projection
- `LabelEncoder` — CIFAR-100 class labels as complex context
- 15% CFG dropout → class-conditional generation at inference
- `predict_meta_physics()` — dynamic Hamiltonian params
- `PhasePool/PhaseUnpool` — 50% FLOPs reduction

*(Note: We use `train_vision_gpu.py` because the underlying JAX code is hardware-agnostic and automatically utilizes the initialized TPU.)*

In [ ]:
import os, json
os.environ['PYTHONPATH'] = '.'
os.environ['JAX_TRACEBACK_FILTERING'] = 'off'

# Load GPU-scaled config
with open('./accel_config.json', 'r') as f:
    accel_cfg = json.load(f)

print(f"🖥️  Training on: {accel_cfg['accel_name']} ({accel_cfg['num_cores']:.1f} cores)")
print(f"📋  batch_size={accel_cfg['batch_size']}  embed_dim={accel_cfg['embed_dim']}  "
      f"base_features={accel_cfg['base_features']}  max_steps={accel_cfg['max_steps']}")

# Run TPU-optimized training
import os; os.system(f"python scripts/train_vision_gpu.py --batch_size {accel_cfg['batch_size']} --embed_dim {accel_cfg['embed_dim']} --base_features {accel_cfg['base_features']} --max_steps {accel_cfg['max_steps']}")


## Cell 5: Monitor Training Loss

Expected loss curve:
- Step 0: ~6.0 (random noise)
- Step 1,000: ~2.0–3.5
- Step 10,000: ~1.2–1.8
- Step 50,000: ~0.8–1.2 (good generative quality)

In [ ]:
import matplotlib.pyplot as plt
import re, os

log_file = 'training_log.txt'
if os.path.exists(log_file):
    steps, losses = [], []
    with open(log_file) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss\s*=\s*([\d.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))

    plt.figure(figsize=(12, 4))
    plt.plot(steps, losses, 'b-', linewidth=1.5, label='Diffusion Loss')
    plt.axhline(y=2.0, color='g', linestyle='--', alpha=0.5, label='Good quality threshold')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('SIFE-LDM V2 Training Curve (CIFAR-100)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('No log file found. Training output is inline above.')

## Cell 6: Generate Images (Unconditional)

Samples from pure noise using the `EulerMaruyamaSampler` with attractor-based early stopping.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

import os; os.system("python scripts/generate_images.py --checkpoint checkpoints/vision/final_vision_model.pkl --num_images 16 --num_steps 100 --output_dir ./generated_images --attractor_stop")

# Display the grid
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('./generated_images/generated_unconditional.png')
plt.figure(figsize=(14, 14))
plt.imshow(img)
plt.axis('off')
plt.title('SIFE-LDM V2 — Unconditional Generation (16 samples)', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 7: Class-Conditional Generation (CFG)

Uses `LabelEncoder` + `cfg_guided_sample()` to steer the SIFE field toward specific CIFAR-100 classes.

**Guidance scale:** Higher = more class-faithful, less variety. 7.5 is a good default.

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
os.environ['PYTHONPATH'] = '.'

# CIFAR-100 classes to generate
CLASSES_TO_GENERATE = {
    69: 'rocket',
    3:  'bear',
    55: 'otter',
    35: 'girl',
}

fig, axes = plt.subplots(1, len(CLASSES_TO_GENERATE), figsize=(16, 5))

for ax, (class_id, class_name) in zip(axes, CLASSES_TO_GENERATE.items()):
    import os; os.system(f"python scripts/generate_images.py --checkpoint checkpoints/vision/final_vision_model.pkl --class_id {class_id} --guidance_scale 7.5 --num_images 4 --num_steps 100 --output_dir ./generated_images")

    img_path = f'./generated_images/generated_class{class_id}_{class_name}.png'
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f'[{class_id}] {class_name}', fontsize=12)
    ax.axis('off')

fig.suptitle('SIFE-LDM V2 — Class-Conditional CFG Generation (s=7.5)', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 8: Attractor Efficiency Benchmark

Measures how many SDE steps the attractor early-stopping saves — the key SIFE physics advantage.

In [ ]:
import sys, os, pickle, jax, jax.numpy as jnp
os.environ['PYTHONPATH'] = '.'
sys.path.insert(0, '.')

from sife.model import SIFELDM, SIFELDMConfig, create_model
from sife.field import SIFEConfig
from sife.diffusion import DiffusionConfig, GaussianDiffusion, EulerMaruyamaSampler
from sife.multiscale import create_multiscale_config

# Load checkpoint
ckpt_path = 'checkpoints/vision/final_vision_model.pkl'
if not os.path.exists(ckpt_path):
    print(f'⚠️ Checkpoint not found at {ckpt_path}. Run training first!')
else:
    with open(ckpt_path, 'rb') as f:
        data = pickle.load(f)
    params = data.params if hasattr(data, 'params') else data

    # Correct Config for V2 Architecture
    config = SIFELDMConfig(
        sife=SIFEConfig(), 
        diffusion=DiffusionConfig(num_timesteps=1000),
        multiscale=create_multiscale_config(num_levels=3, base_features=64),
        is_image=True, 
        image_size=(32, 32), 
        embed_dim=128, 
        batch_size=4,
        num_classes=100  # Enable V2 class conditioning
    )
    
    key = jax.random.PRNGKey(42)
    model, _ = create_model(config, key)
    diffusion = GaussianDiffusion(DiffusionConfig(num_timesteps=1000))
    sampler = EulerMaruyamaSampler(diffusion)

    def model_fn(x, t, context=None):
        return model.apply(params, x, t, context=context, deterministic=True)

    # Run 5 samples and measure steps saved
    shape = (4, 32, 32, 128)
    results = []
    for trial in range(5):
        key, subkey = jax.random.split(key)
        _, steps_used = sampler.sample(
            model_fn, shape, subkey,
            num_steps=100,
            sife_config=config.sife,
            stability_threshold=0.5
        )
        pct_saved = (100 - steps_used) / 100 * 100
        results.append((steps_used, pct_saved))
        print(f'Trial {trial+1}: {steps_used}/100 steps used → {pct_saved:.1f}% saved')

    avg_steps = sum(r[0] for r in results) / len(results)
    avg_saved = sum(r[1] for r in results) / len(results)
    print(f'\nAverage: {avg_steps:.1f} steps, {avg_saved:.1f}% compute saved via attractor stopping')

## Cell 9: Save Output & Cleanup

In [ ]:
import shutil, os, glob

# Zip all generated images for easy download
shutil.make_archive('sife_v2_generated', 'zip', './generated_images')
print('Images zipped to sife_v2_generated.zip — download from Colab Files panel.')

# Optional: clean up intermediate checkpoints to save disk space
intermediate = glob.glob('checkpoints/vision/checkpoint_*.pkl')
for f in intermediate:
    os.remove(f)
print(f'Cleaned {len(intermediate)} intermediate checkpoints. Final model preserved.')